# Practical 3

## Leaderboard

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from nilearn.connectome import sym_matrix_to_vec
from numpy import typing as npt
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

plt.rcParams.update({"font.size": 22, "figure.dpi": 100})

submissions_dir = Path("submissions-3-age-prediction")
test_dir = Path("..") / "ds000117-halfpipe"

We compute the connectivity matrices with the same code as in the practical, using the confounds saved in each checkpoint. The test dataset [ds000117](https://openneuro.org/datasets/ds000117) has nine runs per subject, so we average the connectivity matrices across runs.

In [ ]:
@dataclass
class TimeSeries:
    """
    Represents a time series.

    Attributes:
        path (Path): The path to the ".tsv" file containing the time series.
        metadata (dict[str, Any]): Additional metadata associated with the connectivity matrix.
    """

    path: Path
    confounds_path: Path

    def __post_init__(self) -> None:
        if not self.path.is_file() and self.confounds_path.is_file():
            raise ValueError("Files not found")
    
    @property
    def subject(self) -> str:
        return self.path.name.split("_")[0]
    
    @property
    def values(self) -> npt.NDArray[np.float64]:
        """
        Time series values from the file.

        Returns:
            ndarray: The time series as a NumPy array.
        """
        return np.loadtxt(self.path, delimiter="\t")
    
    @property
    def confounds(self) -> pd.DataFrame:
        """
        Associated confound time series.

        Returns:
            ndarray: The confounds time series as a Pandas data frame.
        """
        return pd.read_table(self.confounds_path)


def calculate_correlation_matrix(time_series: TimeSeries) -> npt.NDArray[np.float64]:
    confounds = time_series.confounds[confounds_columns].to_numpy()
    values = np.nan_to_num(time_series.values)

    if confounds_columns:
        model = make_pipeline(
            SimpleImputer(strategy="constant"), StandardScaler(), LinearRegression()
        )

        predicted_values = model.fit(confounds, values).predict(confounds)
        values = values - predicted_values

    return pd.DataFrame(values).corr().to_numpy()


time_series = [
    TimeSeries(
        path=path,
        confounds_path=path.parent / path.name.replace("timeseries", "desc-confounds_timeseries"),
    )
    for path in sorted(test_dir.rglob("*_run-??_timeseries.tsv"))
]
ages = pd.read_table(test_dir / "participants.tsv").set_index("participant_id").age

In [ ]:
results = []
for path in sorted(submissions_dir.glob("*.joblib")):
    checkpoint = joblib.load(path)
    confounds_columns = checkpoint["confounds_columns"]

    correlation_matrices = np.asarray(
        [calculate_correlation_matrix(time_series) for time_series in tqdm(time_series)]
    )
    X = (
        pd.DataFrame(
            sym_matrix_to_vec(correlation_matrices, discard_diagonal=True),
            index=[time_series.subject for time_series in time_series],
        )
        .groupby(level=0)
        .mean()
    )
    y = ages.loc[X.index]
    predicted = checkpoint["model"].predict(X.to_numpy())

    results.append(
        {
            "name": path.stem.removesuffix(" - checkpoint"),
            "mae": mean_absolute_error(y, predicted),
            "r2": r2_score(y, predicted),
        }
    )

leaderboard = pd.DataFrame(results).sort_values("mae").reset_index(drop=True)
leaderboard.index += 1
leaderboard

Lower mean absolute error is better.

In [ ]:
top = leaderboard.iloc[::-1]

fig, ax = plt.subplots(figsize=(18, 0.9 * len(top) + 2))
colors = ["#c9a227" if rank == 1 else "#4c72b0" for rank in top.index]
bars = ax.barh([f"{rank}.  {name}" for rank, name in top["name"].items()], top["mae"], color=colors)
ax.bar_label(bars, fmt="%.1f years", label_type="center", color="white", weight="bold")

ax.set_xlabel("Mean absolute error (years)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()